# tutorials.hpc_submit_cli

> Hydra-configured HPC submit CLI entry points (bsub batch jobs).

In [ ]:
#| default_exp tutorials.hpc_submit_cli


In [ ]:
#| hide
from nbdev.showdoc import *


In [ ]:
#| export
#| notest
import re
import shlex
import shutil
import subprocess
from datetime import datetime
from pathlib import Path
from typing import Any

import hydra
from omegaconf import DictConfig, OmegaConf

from be_vision_ad_tools.tutorials.end2end_cli import cfg_to_dict


In [ ]:
#| export
def find_project_root(start: Path | None = None) -> Path:
    """Walk parents until we find pyproject.toml with vision_ad_tool metadata."""
    here = (start or Path.cwd()).resolve()
    for parent in [here, *here.parents]:
        pyproject = parent / 'pyproject.toml'
        if pyproject.is_file() and 'vision_ad_tool' in pyproject.read_text(encoding='utf-8'):
            return parent
    return here


In [ ]:
#| export
def _format_override_value(val: Any) -> str:
    if val is None:
        return 'null'
    if isinstance(val, bool):
        return str(val).lower()
    if isinstance(val, (list, tuple)):
        if not val:
            return '[]'
        parts: list[str] = []
        for item in val:
            if isinstance(item, (list, tuple)):
                inner = ','.join(_format_override_value(x) for x in item)
                parts.append(f'[{inner}]')
            else:
                parts.append(_format_override_value(item))
        return f'[{",".join(parts)}]'
    if isinstance(val, str):
        if any(c in val for c in ' \t#=:,[]'):
            return shlex.quote(val)
        return val
    return str(val)


def _dict_to_overrides(d: dict[str, Any], exclude: frozenset[str], prefix: str = '') -> list[str]:
    overrides: list[str] = []
    for key, val in d.items():
        if not prefix and key in exclude:
            continue
        full_key = f'{prefix}.{key}' if prefix else key
        if isinstance(val, dict):
            overrides.extend(_dict_to_overrides(val, exclude, full_key))
        else:
            overrides.append(f'{full_key}={_format_override_value(val)}')
    return overrides


def cfg_to_hydra_overrides(cfg: DictConfig, exclude: frozenset[str] = frozenset({'hpc'})) -> list[str]:
    """Convert resolved Hydra config to CLI override strings (excludes HPC section)."""
    container = cfg_to_dict(cfg)
    return _dict_to_overrides(container, exclude)


In [ ]:
#| export
def build_bsub_args(hpc: dict[str, Any]) -> dict[str, str]:
    """Build bsub flag dict from resolved ``hpc`` config section."""
    from be_vision_ad_tools.inference.multinode_from_aiop_tool import HPC_Job

    if hpc.get('use_gpu'):
        bsub_args = HPC_Job.BSUB_ARGS_DEFAULT_GPU.copy()
        bsub_args['-q'] = hpc.get('queue', 'gpu')
        resource = hpc.get('gpu_resource_select', 'osrel>=70 && type=any')
        gpu_spec = hpc.get('gpu_spec', 'num=1:j_exclusive=yes')
        bsub_args['-gpu'] = gpu_spec
    else:
        bsub_args = HPC_Job.BSUB_ARGS_DEFAULT.copy()
        bsub_args['-q'] = hpc.get('queue', 'batch')
        resource = hpc.get('resource_select', bsub_args.get('-R', ''))

    mem = hpc.get('memory_mb')
    if mem:
        resource = f'{resource} && rusage[mem={mem}]'
    bsub_args['-R'] = resource

    walltime = hpc.get('walltime_minutes')
    if walltime is not None:
        bsub_args['-W'] = str(walltime)

    bsub_args['-n'] = str(hpc.get('cores', 4))
    return bsub_args


In [ ]:
#| export
def build_remote_shell_command(hpc: dict[str, Any], overrides: list[str]) -> str:
    """Build shell command that runs the local Hydra CLI with overrides on the cluster."""
    project_root = hpc.get('project_root')
    root = Path(project_root).resolve() if project_root else find_project_root()
    run_prefix = hpc.get('run_prefix')
    if run_prefix is None:
        run_prefix = 'uv run' if shutil.which('uv') else ''

    local_cli = hpc.get('local_cli')
    if not local_cli:
        raise ValueError('hpc.local_cli is required')

    parts: list[str] = []
    if run_prefix:
        parts.extend(run_prefix.split())
    parts.append(local_cli)
    parts.extend(overrides)
    inner = ' '.join(parts)
    return f'cd {shlex.quote(str(root))} && {inner}'


In [ ]:
#| export
def submit_hydra_job(cfg: DictConfig) -> dict[str, Any]:
    """Submit a batch job via ``bsub`` that runs the paired local Hydra CLI."""
    hpc = dict(cfg_to_dict(cfg).get('hpc', {}))
    overrides = cfg_to_hydra_overrides(cfg)
    shell_cmd = build_remote_shell_command(hpc, overrides)
    bsub_flags = build_bsub_args(hpc)

    job_name = hpc.get('job_name', 'vad-job')
    log_dir = Path(hpc.get('log_dir', './hpc_logs'))
    log_dir.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    stdout = log_dir / f'{job_name}_{stamp}{hpc.get("stdout_suffix", ".out")}'
    stderr = log_dir / f'{job_name}_{stamp}{hpc.get("stderr_suffix", ".err")}'

    bsub_cmd: list[str] = ['bsub', '-J', job_name, '-oo', str(stdout), '-eo', str(stderr)]
    for flag, value in bsub_flags.items():
        bsub_cmd.extend([flag, value])
    bsub_cmd.extend(['/bin/bash', '-lc', shell_cmd])

    result: dict[str, Any] = {
        'job_name': job_name,
        'log_stdout': str(stdout),
        'log_stderr': str(stderr),
        'remote_command': shell_cmd,
        'bsub_argv': bsub_cmd,
    }

    if hpc.get('dry_run'):
        result['dry_run'] = True
        result['message'] = 'dry_run: bsub not invoked'
        print(result)
        return result

    if shutil.which('bsub') is None:
        result['error'] = 'bsub not found on PATH — set hpc.dry_run=true to preview submission'
        print(result)
        return result

    proc = subprocess.run(bsub_cmd, capture_output=True, text=True)
    result['returncode'] = proc.returncode
    result['stdout'] = proc.stdout.strip()
    result['stderr'] = proc.stderr.strip()
    match = re.search(r'Job <(\d+)>', proc.stdout + proc.stderr)
    if match:
        result['lsf_job_id'] = int(match.group(1))
    print(result)
    return result


In [ ]:
#| export
_CONF = str(Path(__file__).resolve().parent / 'conf')


def compose_submit_cfg(config_name: str, overrides: list[str] | None = None) -> DictConfig:
    """Compose a submit config with Hydra defaults plus optional overrides."""
    from hydra import compose, initialize_config_dir

    with initialize_config_dir(version_base=None, config_dir=_CONF):
        return compose(config_name=config_name, overrides=overrides or [])


In [ ]:
#| export
@hydra.main(version_base=None, config_path=_CONF, config_name='train_submit')
def train_submit_cli(cfg: DictConfig) -> None:
    """Submit vad-train to HPC via bsub."""
    submit_hydra_job(cfg)


@hydra.main(version_base=None, config_path=_CONF, config_name='infer_submit')
def infer_submit_cli(cfg: DictConfig) -> None:
    """Submit vad-infer to HPC via bsub."""
    submit_hydra_job(cfg)


@hydra.main(version_base=None, config_path=_CONF, config_name='organize_submit')
def organize_submit_cli(cfg: DictConfig) -> None:
    """Submit vad-organize to HPC via bsub."""
    submit_hydra_job(cfg)


@hydra.main(version_base=None, config_path=_CONF, config_name='infer_organize_submit')
def infer_organize_submit_cli(cfg: DictConfig) -> None:
    """Submit vad-infer-organize to HPC via bsub."""
    submit_hydra_job(cfg)


@hydra.main(version_base=None, config_path=_CONF, config_name='train_infer_submit')
def train_infer_submit_cli(cfg: DictConfig) -> None:
    """Submit vad-train-infer to HPC via bsub."""
    submit_hydra_job(cfg)


@hydra.main(version_base=None, config_path=_CONF, config_name='full_submit')
def full_submit_cli(cfg: DictConfig) -> None:
    """Submit vad-full to HPC via bsub."""
    submit_hydra_job(cfg)


@hydra.main(version_base=None, config_path=_CONF, config_name='hyperparam_search_submit')
def hyperparam_search_submit_cli(cfg: DictConfig) -> None:
    """Submit vad-hyperparam-search to HPC via bsub."""
    submit_hydra_job(cfg)
